# 04 — ML Pipelines & Hyperparameter Tuning

Interview-oriented notes + practical sklearn code.

**Covers:** Pipeline, ColumnTransformer, preprocessing leakage, cross-validation, GridSearchCV, RandomizedSearchCV, Bayesian optimization (Optuna), tuning Pipeline parameters, model comparison, validation strategy, nested CV concept, early stopping concept, learning curves, metrics, imbalance, computational cost, `n_jobs`, reproducibility, saving/loading pipelines, production flow, common mistakes, and interview cheat sheet.

In [1]:
# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score,
    learning_curve, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import randint, uniform, loguniform

## 2. Pipeline Internal Flow

**Training**

`X_train → scaler.fit_transform() → model.fit()`

**Prediction**

`X_test → scaler.transform() → model.predict()`

Never fit preprocessing on the test set.

In [2]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

print(pipeline.named_steps)


{'scaler': StandardScaler(), 'model': LogisticRegression(max_iter=2000)}


## 3. Pipeline + Cross Validation

For every CV fold, preprocessing is fitted only on that fold's training portion. This is a major reason Pipeline is important.

In [3]:
X, y = load_breast_cancer(return_X_y=True)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    pipeline, X, y, cv=cv, scoring="accuracy"
)

print("Fold scores:", scores)
print("Mean:", scores.mean())
print("Std:", scores.std())


Fold scores: [0.97368421 0.94736842 0.96491228 0.99122807 0.99115044]
Mean: 0.9736686849868033
Std: 0.016627222216787987


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

## 4. ColumnTransformer + Pipeline

Real datasets commonly contain numerical, categorical, and missing values. Different preprocessing can be applied to different columns.

In [4]:
df = pd.DataFrame({
    "age": [22, 30, 40, np.nan, 35, 28, 45, 32, 52, 26],
    "salary": [30000, 50000, 80000, 120000, 70000,
               40000, 100000, 60000, 140000, 35000],
    "city": ["Chennai", "Mumbai", "Delhi", "Chennai", "Mumbai",
             "Delhi", "Chennai", "Mumbai", "Delhi", "Chennai"],
    "target": [0, 0, 1, 1, 1, 0, 1, 0, 1, 0]
})

X = df.drop("target", axis=1)
y = df["target"]

numeric_features = ["age", "salary"]
categorical_features = ["city"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

model_pipeline.fit(X, y)
print(model_pipeline.predict(X))


[0 0 1 1 1 0 1 0 1 0]


## 5. Hyperparameter Tuning

**Parameters** are learned from data.

**Hyperparameters** are chosen externally.

Examples:
- Decision Tree → `max_depth`
- Random Forest → `n_estimators`
- Logistic Regression → `C`
- Ridge/Lasso → `alpha`

Tuning means trying configurations and selecting one using validation/CV performance.

## 6. GridSearchCV

GridSearch evaluates **every combination** in the supplied grid.

Example: 3 values × 3 values = 9 configurations. With 5-fold CV, that is 45 fits.

In [5]:
X, y = load_breast_cancer(return_X_y=True)

model = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [2, 4, 6, 8, None],
    "min_samples_split": [2, 5, 10]
}

grid_search = GridSearchCV(
    model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X, y)

print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)
best_model = grid_search.best_estimator_


Best parameters: {'max_depth': 2, 'min_samples_split': 2}
Best CV score: 0.9279614966620089


## 7. RandomizedSearchCV

RandomizedSearch samples a fixed number of configurations.

Useful when the search space is large and exhaustive GridSearch is expensive.

In [6]:
X, y = load_breast_cancer(return_X_y=True)

model = RandomForestClassifier(random_state=42)

param_distributions = {
    "n_estimators": randint(50, 300),
    "max_depth": randint(2, 20),
    "min_samples_split": randint(2, 20),
    "max_features": ["sqrt", "log2", None]
}

random_search = RandomizedSearchCV(
    model,
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X, y)

print("Best parameters:", random_search.best_params_)
print("Best CV score:", random_search.best_score_)


Best parameters: {'max_depth': 9, 'max_features': 'sqrt', 'min_samples_split': 8, 'n_estimators': 171}
Best CV score: 0.9631113181183046


## 8. Grid Search vs Random Search

**GridSearch:** exhaustive over the supplied grid.

**RandomizedSearch:** samples configurations.

Interview:
> GridSearchCV exhaustively evaluates a predefined parameter grid, while RandomizedSearchCV samples a fixed number of configurations from the search space.

## 9. Tuning a Pipeline

Use:

`step_name__parameter_name`

Examples:
- `model__C`
- `scaler__with_mean`

In [7]:
X, y = load_breast_cancer(return_X_y=True)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

param_grid = {
    "scaler__with_mean": [True, False],
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1
)

grid.fit(X, y)

print("Best parameters:", grid.best_params_)
print("Best score:", grid.best_score_)


Best parameters: {'model__C': 1, 'scaler__with_mean': True}
Best score: 0.9806862288464524


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

## 10. Tuning Different Models

The model itself can be a hyperparameter. This allows comparison of different algorithms inside one search workflow.

In [8]:
X, y = load_breast_cancer(return_X_y=True)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

param_grid = [
    {
        "model": [LogisticRegression(max_iter=2000)],
        "model__C": [0.1, 1, 10]
    },
    {
        "model": [DecisionTreeClassifier(random_state=42)],
        "model__max_depth": [2, 4, 6, 10]
    }
]

grid = GridSearchCV(
    pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1
)

grid.fit(X, y)

print(grid.best_params_)
print(grid.best_score_)


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

{'model': LogisticRegression(max_iter=2000), 'model__C': 1}
0.9806862288464524


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

## 11. Important Hyperparameters

### Logistic Regression
- `C`: inverse regularization strength.
- Small `C` → stronger regularization.
- Large `C` → weaker regularization.
- `penalty`: regularization type.

### Decision Tree
- `max_depth`: maximum depth.
- `min_samples_split`: minimum samples needed to split.
- `min_samples_leaf`: minimum samples in a leaf.
- `criterion`: split quality.

### Random Forest
- `n_estimators`: number of trees.
- `max_depth`: maximum tree depth.
- `max_features`: features considered at each split.
- `min_samples_split`
- `min_samples_leaf`

## 12. Parameters vs Hyperparameters

`RandomForestClassifier(n_estimators=200)`

`n_estimators` is a hyperparameter.

The learned tree structures, splits, and thresholds are learned from training data.

**Interview:** Hyperparameters configure the learning algorithm; parameters are learned from training data.

## 13. Correct Validation Strategy

```text
DATA
 |
 +-- TRAIN ----------------------+
 |                                |
 |                         Cross Validation
 |                                |
 |                       Hyperparameter Tuning
 |                                |
 |                           Best Model
 |                                |
 +-- TEST <-----------------------+
          Final evaluation
```

The test set should remain untouched until final evaluation.

## 14. Complete Tuning Workflow

1. Split train/test.
2. Build preprocessing.
3. Build Pipeline.
4. Define CV.
5. Define search space.
6. Run GridSearchCV or RandomizedSearchCV.
7. Select best configuration.
8. Evaluate once on held-out test data.

In [9]:
X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"]
}

grid = GridSearchCV(
    pipeline, param_grid, cv=5, scoring="f1", n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

test_pred = grid.best_estimator_.predict(X_test)
print("Final test F1:", f1_score(y_test, test_pred))


Best parameters: {'model__C': 0.1, 'model__penalty': 'l2'}
Best CV F1: 0.984524686809138
Final test F1: 0.9793103448275862


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

## 15. Why Test Data Is Not Used During Tuning

If we repeatedly use test performance to choose models, the test set influences model selection.

Then it is no longer an unbiased final estimate.

Correct:

`TRAIN → CV → TUNING → BEST MODEL → TEST ONCE`

## 16. Nested Cross Validation

Nested CV separates:

- **Inner CV:** hyperparameter tuning.
- **Outer CV:** estimates generalization.

```text
Outer Fold
   |
   +-- Inner CV → tune parameters
   |
   +-- Evaluate tuned model
```

Know the concept for interviews. You generally do not need to implement it from scratch.

## 17. Early Stopping

Some iterative algorithms stop when validation performance stops improving.

`Train → Monitor validation → Improving? Continue : Stop`

Common in gradient boosting, XGBoost, LightGBM, and neural networks.

Benefits:
- Saves computation.
- Can reduce overfitting.

## 18. Learning Curves

Learning curves show training and validation performance as training-set size increases.

Useful for diagnosing:
- High bias
- High variance
- Whether more training data may help.

In [10]:
X, y = load_breast_cancer(return_X_y=True)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

train_sizes, train_scores, validation_scores = learning_curve(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy",
    train_sizes=np.linspace(0.1, 1.0, 5)
)

print("Training scores:", train_scores.mean(axis=1))
print("Validation scores:", validation_scores.mean(axis=1))


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

Training scores: [1.         0.98367347 0.9816     0.98522727 0.98945055]
Validation scores: [0.79251669 0.94725974 0.95958702 0.97011334 0.98068623]


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.1

## 19. Underfitting vs Overfitting During Tuning

For a Decision Tree:
- Very small `max_depth` → potentially underfit.
- Moderate depth → potentially better generalization.
- Very large/unlimited depth → potentially overfit.

Tuning searches for useful model complexity.

In [11]:
X, y = load_breast_cancer(return_X_y=True)

for depth in [1, 2, 4, 8, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)

    scores = cross_val_score(
        model, X, y, cv=5, scoring="accuracy"
    )

    print("depth =", depth, "CV =", scores.mean())


depth = 1 CV = 0.8998447446048751


depth = 2 CV = 0.9279614966620089
depth = 4 CV = 0.9208818506443098


depth = 8 CV = 0.915587641670548
depth = None CV = 0.9173420276354604


## 20. Multiple Scoring Metrics

Metric choice depends on the problem.

Possible metrics:
- Accuracy
- Precision
- Recall
- F1
- ROC-AUC
- PR-AUC / Average Precision

For imbalanced problems, accuracy alone can be misleading.

In [12]:
X, y = load_breast_cancer(return_X_y=True)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

grid = GridSearchCV(
    pipeline,
    {"model__C": [0.01, 0.1, 1, 10]},
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X, y)

print("Best parameters:", grid.best_params_)
print("Best F1:", grid.best_score_)


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

Best parameters: {'model__C': 1}
Best F1: 0.9847673174099155


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

## 21. Imbalanced Data During Tuning

For imbalanced classification, consider precision, recall, F1, ROC-AUC, or average precision instead of blindly optimizing accuracy.

`class_weight="balanced"` can also be useful for supported models.

In [13]:
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    weights=[0.95, 0.05],
    random_state=42
)

model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000
)

scores = cross_val_score(
    model, X, y, cv=5, scoring="f1"
)

print("F1 scores:", scores)
print("Mean F1:", scores.mean())


F1 scores: [0.43373494 0.38636364 0.37623762 0.49350649 0.42696629]
Mean F1: 0.4233617971052747


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

## 22. Random Search with Continuous Values

RandomizedSearchCV can sample values from distributions instead of only trying a fixed list.

In [14]:
X, y = load_breast_cancer(return_X_y=True)

model = LogisticRegression(max_iter=2000)

param_distributions = {
    # C matters on a log scale (0.001, 0.01, 0.1, 1, 10 ...), so sample it log-uniformly.
    # Note: scipy's uniform(a, b) means loc=a, scale=b, i.e. [a, a+b], not [a, b].
    "C": loguniform(1e-3, 1e2)
}

random_search = RandomizedSearchCV(
    model,
    param_distributions,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X, y)

print(random_search.best_params_)
print(random_search.best_score_)


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/u

/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/u

/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/u

... (10 outputs left out)

/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: inva

/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/vivek-18890/Python/.venv310/lib/python3.1

{'C': np.float64(70.72114131472235)}
0.9560937742586555


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 23. Bayesian Optimization with Optuna

Grid and random search choose their trials blindly. **Bayesian optimization** learns from the trials so far: it models
"score as a function of the hyperparameters" and picks the next trial where an improvement looks likely.
Optuna, the most common library for this, uses TPE (Tree-structured Parzen Estimator) by default, and it can **prune**:
stop a bad trial early from its intermediate scores, for example after a few epochs or boosting rounds.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)

import numpy as np
import optuna
from scipy.stats import loguniform, randint
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)
X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.25, stratify=y_bc, random_state=42)
cv = StratifiedKFold(5, shuffle=True, random_state=0)


def objective(trial):
    model = HistGradientBoostingClassifier(
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.5, log=True),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 4, 64, log=True),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 60),
        l2_regularization=trial.suggest_float("l2_regularization", 1e-4, 10, log=True),
        random_state=0)
    return cross_val_score(model, X_tr, y_tr, cv=cv).mean()


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=30)

random_search = RandomizedSearchCV(
    HistGradientBoostingClassifier(random_state=0),
    {"learning_rate": loguniform(0.01, 0.5), "max_leaf_nodes": randint(4, 65),
     "min_samples_leaf": randint(5, 61), "l2_regularization": loguniform(1e-4, 10)},
    n_iter=30, cv=cv, random_state=0).fit(X_tr, y_tr)

default = cross_val_score(HistGradientBoostingClassifier(random_state=0), X_tr, y_tr, cv=cv).mean()
optuna_best = np.maximum.accumulate([t.value for t in study.trials])
random_best = np.maximum.accumulate(random_search.cv_results_["mean_test_score"])
print(f"default settings:  CV accuracy {default:.4f}")
print(f"best after 10 / 20 / 30 trials   Optuna: {optuna_best[[9, 19, 29]].round(4)}   random search: {random_best[[9, 19, 29]].round(4)}")
print("Optuna's best settings:", {k: round(v, 4) if isinstance(v, float) else v for k, v in study.best_params.items()})


default settings:  CV accuracy 0.9578
best after 10 / 20 / 30 trials   Optuna: [0.9648 0.9671 0.9672]   random search: [0.9648 0.9648 0.9672]
Optuna's best settings: {'learning_rate': 0.3175, 'max_leaf_nodes': 5, 'min_samples_leaf': 45, 'l2_regularization': 7.8645}


Both searches beat the defaults (0.9578) and end at the same best CV score, 0.9672. Optuna was there after 20 trials (0.9671); random search was still at 0.9648
and needed all 30. On a small, cheap problem like this the difference is small. It grows when each trial is expensive (deep learning, big data)
and the search space is large, which is where Bayesian optimization is used.

| | Grid search | Random search | Bayesian optimization (Optuna) |
|---|---|---|---|
| Picks trials | every combination | at random | from how earlier trials scored |
| Gain | exhaustive on a small grid; easy to explain | cheap; covers continuous ranges well; fully parallel | fewest trials for a good result; prunes bad trials early |
| Cost | explodes with every extra parameter | wastes trials in bad regions | sequential by nature (harder to parallelize); more setup |
| Pick it when | 1–2 parameters with few values | many parameters, cheap trials | expensive trials: deep learning, large data, long pipelines |

## 24. Computational Cost

Suppose:
- 10 values for A
- 5 values for B
- 3 model choices
- 5 CV folds

Fits = `10 × 5 × 3 × 5 = 750`

Ways to reduce cost:
- RandomizedSearchCV
- Smaller search space
- Fewer CV folds when appropriate
- Better parameter ranges
- Parallel processing
- Early stopping
- Coarse-to-fine tuning

## 25. `n_jobs`

`n_jobs` controls parallel execution for supported sklearn operations.

`n_jobs=-1` means use all available CPU cores.

Trade-off: faster execution can mean higher CPU and memory usage.

In [15]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print(model.get_params()["n_jobs"])


-1


## 26. `random_state`

`random_state` makes randomized operations reproducible.

It does not make a model smarter or automatically improve accuracy.

In [16]:
model1 = RandomForestClassifier(
    n_estimators=100, random_state=42
)

model2 = RandomForestClassifier(
    n_estimators=100, random_state=42
)

print(model1.get_params()["random_state"])
print(model2.get_params()["random_state"])


42
42


## 27. Save the Complete Pipeline

Save the complete preprocessing + model pipeline.

Why? Preprocessing is part of the model system.

`Raw Input → Imputer → Encoder → Scaler → Model`

In [17]:
X, y = load_breast_cancer(return_X_y=True)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

pipeline.fit(X, y)

joblib.dump(pipeline, "ml_pipeline.joblib")

print("Pipeline saved.")


Pipeline saved.


/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

In [18]:
loaded_pipeline = joblib.load("ml_pipeline.joblib")

pred = loaded_pipeline.predict(X[:5])

print("Predictions:", pred)


Predictions: [0 0 0 0 0]


## 28. Production Inference Flow

### Training

`Raw Training Data → Preprocessing → Feature Engineering → Tuning → Final Pipeline → Save`

### Inference

`New Raw Data → Same Preprocessing → Same Feature Engineering → Trained Model → Prediction`

Training and inference must use consistent transformations.

## 29. Common Mistakes

1. Scaling before train/test split.
2. Fitting encoders on the entire dataset.
3. Using the test set for tuning.
4. Optimizing accuracy blindly for imbalanced data.
5. Running huge GridSearch without estimating cost.
6. Not using Pipeline with preprocessing during CV.
7. Saving only the classifier.
8. Creating features using future information.
9. Searching unnecessarily huge parameter spaces.
10. Treating CV score as the final test score.

## 30. Interview Cheat Sheet

**Why Pipeline?**  
Combines preprocessing and modeling, prevents preprocessing leakage during CV, and keeps training/inference consistent.

**GridSearchCV vs RandomizedSearchCV?**  
GridSearch evaluates every supplied combination. RandomizedSearch samples a fixed number of configurations.

**Why not tune on the test set?**  
Because test feedback becomes part of model selection and causes optimistic evaluation.

**What does `cv=5` mean?**  
Five-fold cross-validation.

**What does `n_jobs=-1` mean?**  
Use all available CPU cores for supported parallel operations.

**What does `C` mean in LogisticRegression?**  
Inverse regularization strength. Small C means stronger regularization.

**Why RandomizedSearch?**  
It can explore a large space with fewer fits.

**Why save the Pipeline?**  
So inference applies the same preprocessing as training.

**Parameter vs hyperparameter?**  
Parameters are learned from data; hyperparameters configure the learning algorithm.

**Correct workflow?**

`Train/Test Split → CV on Training Data → Hyperparameter Tuning → Best Model → Final Test Evaluation`

# 31. Final Mental Model

```text
                    RAW DATA
                        |
                        ↓
                TRAIN / TEST SPLIT
                   /          \
                  /            \
              TRAIN            TEST
                |
                ↓
             PIPELINE
                |
        ┌───────┴────────┐
        ↓                ↓
  PREPROCESSING        MODEL
        |                |
        └───────┬────────┘
                ↓
        CROSS VALIDATION
                ↓
      HYPERPARAMETER SEARCH
          /           \
         /             \
   GridSearch       RandomSearch
         \             /
          \           /
             ↓
         BEST MODEL
             |
             ↓
       FINAL TEST SET
             |
             ↓
          METRICS
             |
             ↓
       SAVE PIPELINE
             |
             ↓
          DEPLOY
```

### Interview depth for this module

**Know deeply**
- Pipeline purpose and internal flow
- Cross-validation
- Leakage prevention
- Train/CV/Test separation
- Grid vs Random Search
- Hyperparameter vs parameter
- Search computational cost
- Production consistency

**Know practically**
- `Pipeline`
- `ColumnTransformer`
- `cross_val_score`
- `GridSearchCV`
- `RandomizedSearchCV`
- `learning_curve`
- `joblib`

You do **not** need to implement sklearn's Pipeline/GridSearch from scratch.